# 10 — Comparação final e submissão

Carrega os resultados dos notebooks 03–09 sem repetir os treinamentos. A escolha usa apenas os resultados externos; o teste é lido somente depois, para prever. Execute antes as famílias que deseja comparar em modo completo.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src" / "data.py").is_file():
    raise RuntimeError("Abra o notebook a partir da raiz do projeto ou da pasta notebooks.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data import load_train
from src.experiments import (recipes, evaluate_family, summary, save_results,
                             plot_diagnostics, load_results, search)
X, y = load_train()
pd.set_option("display.precision", 4)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
print(f"Treino: {len(y)} pares; {X.shape[1]} atributos brutos.")

In [ ]:
payloads = load_results(X, y)
expected = {"lineares", "splines", "knn", "arvores", "boosting", "svr", "mlp"}
missing = expected - {p["family"] for p in payloads}
print("Famílias pendentes:", sorted(missing) or "nenhuma")
ranking = pd.concat([summary(p) for p in payloads], ignore_index=True)
ranking = ranking.drop_duplicates(subset=["modelo"]).sort_values("RMSE validação").reset_index(drop=True)
display(ranking)

## Comparação pareada

Comparamos o candidato de menor RMSE com a linear com afinidade nas mesmas dobras. O desvio das diferenças descreve a variação; a dependência entre dobras impede interpretá-lo como evidência formal de significância.

A validação aninhada de cada receita protege a escolha de hiperparâmetros, mas escolher a melhor família olhando esta tabela ainda pode introduzir otimismo. Uma avaliação independente ou um laço externo envolvendo também a escolha de família seria necessário para medir o procedimento completo.

In [ ]:
winner = ranking.iloc[0]
selected_payload = next(p for p in payloads if p["family"] == winner["família"])
selected_record = next(r for r in selected_payload["records"] if r["name"] == winner["modelo"])
baseline = next(r for r in selected_payload["records"] if r["name"] == "Linear + afinidade")
difference = np.array(selected_record["val_scores"]) - np.array(baseline["val_scores"])
print(f"Candidato: {winner['modelo']}")
print(f"Diferença média para a linear: {difference.mean():.4f}")
print(f"Desvio das diferenças: {difference.std(ddof=1):.4f}")
print(f"Melhor em {(difference < 0).sum()} de {len(difference)} dobras")
ranking.set_index("modelo")[["RMSE treino (E_in)", "RMSE validação"]].plot.barh(figsize=(8, 6))
plt.xlabel("RMSE")
plt.tight_layout()
plt.show()

## Reajuste no treino completo

Por padrão, selecionamos o primeiro colocado. Você pode mudar `SELECTED_FAMILY` e `SELECTED_MODEL` para preferir um modelo mais simples. Uma nova busca interna usa todo o treino disponível, sem consultar o teste. Esta busca serve ao ajuste final; seu melhor escore interno não substitui os resultados externos.

In [ ]:
SELECTED_FAMILY = winner["família"]
SELECTED_MODEL = winner["modelo"]

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import FunctionTransformer
from src.features import raw_plus_affinity
from src.experiments import pipeline

if SELECTED_MODEL == "Constante":
    model, grid = DummyRegressor(), {}
elif SELECTED_MODEL == "Linear + afinidade":
    model, grid = pipeline(LinearRegression(), FunctionTransformer(raw_plus_affinity)), {}
else:
    model, grid = recipes(SELECTED_FAMILY)[SELECTED_MODEL]
final_search = search(model, grid).fit(X, y)
print("Configuração final:", final_search.best_params_)

## Previsão e exportação

Adicione `data/test.csv` com `pair_id` e os mesmos atributos do treino. A célula abaixo preserva a ordem dos pares e grava notas reais em [0, 5], sem arredondamento. Se o teste ainda não estiver disponível, a exportação fica pendente.

In [ ]:
from src.data import load_test

if (ROOT / "data" / "test.csv").is_file():
    ids, X_test = load_test()
    submission = pd.DataFrame({"pair_id": ids, "stars": np.clip(final_search.predict(X_test), 0, 5)})
    assert len(submission) == len(X_test) and submission["stars"].notna().all()
    output_dir = ROOT / "submissions"
    output_dir.mkdir(exist_ok=True)
    output = output_dir / "submission_final.csv"
    submission.to_csv(output, index=False)
    print("Submissão salva em:", output)
    display(submission.head())
else:
    print("Exportação pendente: adicione data/test.csv e execute novamente esta célula.")

## Conclusão para o relatório

Registre o modelo escolhido, configuração final, RMSE externo, comparação com os baselines, custo e limitações da seleção entre famílias. Se alguma família não foi executada, explicite o escopo da comparação. Não use o desempenho no teste para voltar a selecionar modelos.